In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# 1. Load All Data 

In [2]:
comp_df = pd.read_csv('train.csv')
original_df = pd.read_csv('synthetic_road_accidents_100k.csv')
test_df = pd.read_csv('test.csv')

# Saved the test IDs now so we don't lose them
test_ids = test_df['id']

# 2.Prepare Training and Test Data

In [3]:
# Combine the training data
comp_df = comp_df.drop('id', axis=1) 
comp_df['is_original'] = 0
original_df['is_original'] = 1
full_train_df = pd.concat([comp_df, original_df], ignore_index=True)

# List of our dataframes to apply feature engineering to
datasets = [full_train_df, test_df]

# Applying the same transformations to both train and test data
for df in datasets:
    # credit goes to @tilii for this feature
    df['gm_risk_feature'] = (
        0.3 * df["curvature"] + 0.2 * (df["lighting"] == "night").astype(int) +
        0.1 * (df["weather"] != "clear").astype(int) + 0.2 * (df["speed_limit"] >= 60).astype(int) +
        0.1 * (np.array(df["num_reported_accidents"]) > 2).astype(int)
    )
    
    # Convert boolean columns to integers for consistency
    for col in df.select_dtypes(include='bool').columns:
        df[col] = df[col].astype(int)

print("Feature Engineering Complete!")

Feature Engineering Complete!


# 3. Encoding and Final Preparation

In [4]:
# Separate target variable from the full training data
X = full_train_df.drop('accident_risk', axis=1)
y = full_train_df['accident_risk']

# Drop the ID from the test features (we already saved test_ids)
test_features = test_df.drop('id', axis=1)

# Use One-Hot Encoding for all text-based columns
X = pd.get_dummies(X, columns=X.select_dtypes(include='object').columns, drop_first=True)
test_features = pd.get_dummies(test_features, columns=test_features.select_dtypes(include='object').columns, drop_first=True)

# Align columns to ensure train and test sets match perfectly
X_aligned, test_aligned = X.align(test_features, join='left', axis=1, fill_value=0)

print("Data is now fully prepared and aligned for modeling.")

Data is now fully prepared and aligned for modeling.


# 4. Validate Our Blended Model with Cross-Validation 

In [5]:
from sklearn.metrics import root_mean_squared_error
kf = KFold(n_splits=5, shuffle=True, random_state=42)
blended_rmse_scores = []
print("\n--- Starting Blended Cross-Validation ---")

for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
    X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Train both models
    xgb_model = XGBRegressor(random_state=42).fit(X_train, y_train)
    lgbm_model = LGBMRegressor(random_state=42).fit(X_train, y_train)
    
    # Blend predictions
    xgb_preds = xgb_model.predict(X_val)
    lgbm_preds = lgbm_model.predict(X_val)
    blended_preds = (xgb_preds * 0.5) + (lgbm_preds * 0.5)
    
    rmse = root_mean_squared_error(y_val, blended_preds)
    blended_rmse_scores.append(rmse)

print(f"\nAverage Blended CV RMSE: {np.mean(blended_rmse_scores)}")


--- Starting Blended Cross-Validation ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023983 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 428
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.357184
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024602 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 431
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.357290
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020777 seconds.
You can set `force_row_wise=true` to remo

# 5. tuning xgboost model with optuna

In [7]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

# We'll use a single train/validation split for faster tuning
X_train, X_val, y_train, y_val = train_test_split(X_aligned, y, test_size=0.2, random_state=42)

# 1. Define the "objective" function for Optuna
# This function tells Optuna what to measure (our RMSE)
def objective(trial):
    # 2. Define the search space for the hyperparameters
    params = {
        'objective': 'reg:squarederror',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    
    # Train the model with the suggested parameters
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    
    # Make a prediction and calculate the RMSE
    preds = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, preds)
    
    return rmse

# 3. Create and run the study
# n_trials=25 means Optuna will run the experiment 25 times
study = optuna.create_study(direction='minimize') # 'minimize' because we want the lowest RMSE
study.optimize(objective, n_trials=25)

# Print the best results
print("\n--- Optuna Tuning Results ---")
print(f"Best trial RMSE: {study.best_value}")
print("Best hyperparameters found:")
print(study.best_params)

[I 2025-10-23 11:30:34,521] A new study created in memory with name: no-name-1a779212-ef88-497b-85c5-b678fe4187ae
[I 2025-10-23 11:31:30,904] Trial 0 finished with value: 0.05604998797317884 and parameters: {'n_estimators': 646, 'learning_rate': 0.2980557981895813, 'max_depth': 6, 'subsample': 0.6833057732457527, 'colsample_bytree': 0.6048186209878091}. Best is trial 0 with value: 0.05604998797317884.
[I 2025-10-23 11:31:42,853] Trial 1 finished with value: 0.08195667166456401 and parameters: {'n_estimators': 100, 'learning_rate': 0.010949403069773744, 'max_depth': 7, 'subsample': 0.8647751116151046, 'colsample_bytree': 0.6941358730865292}. Best is trial 0 with value: 0.05604998797317884.
[I 2025-10-23 11:32:21,209] Trial 2 finished with value: 0.05588645851870846 and parameters: {'n_estimators': 698, 'learning_rate': 0.1726407599868586, 'max_depth': 3, 'subsample': 0.6346409013745671, 'colsample_bytree': 0.6595234328869763}. Best is trial 2 with value: 0.05588645851870846.
[I 2025-10-


--- Optuna Tuning Results ---
Best trial RMSE: 0.05559072851621799
Best hyperparameters found:
{'n_estimators': 323, 'learning_rate': 0.1410864023683022, 'max_depth': 6, 'subsample': 0.9020911120316853, 'colsample_bytree': 0.8370745242901517}


## first run of optuna rmse score was better with these param so we will use this instead of above

In [8]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np

# These are the best parameters we just found with Optuna
tuned_xgb_params = {
    'n_estimators': 617,
    'learning_rate': 0.016363153740373525,
    'max_depth': 8,
    'subsample': 0.8541055988151639,
    'colsample_bytree': 0.7475747293118165,
    'objective': 'reg:squarederror',
    'random_state': 42
}

# Set up the K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("--- Starting Cross-Validation for Tuned XGBoost Model ---")

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
    print(f"--- Fold {fold+1} ---")
    
    X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Initialize the model WITH OUR TUNED PARAMETERS
    tuned_model = XGBRegressor(**tuned_xgb_params)
    tuned_model.fit(X_train, y_train)
    
    y_pred = tuned_model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    
    rmse_scores.append(rmse)
    print(f"RMSE for Fold {fold+1}: {rmse}")

# Calculate the final average CV score
print(f"\nAverage CV RMSE for Tuned XGBoost: {rmse_scores}")

--- Starting Cross-Validation for Tuned XGBoost Model ---
--- Fold 1 ---
RMSE for Fold 1: 0.05558269899016783
--- Fold 2 ---
RMSE for Fold 2: 0.05501567646043786
--- Fold 3 ---
RMSE for Fold 3: 0.055021240135226926
--- Fold 4 ---
RMSE for Fold 4: 0.05504638818988694
--- Fold 5 ---
RMSE for Fold 5: 0.05488723769555776

Average CV RMSE for Tuned XGBoost: [0.05558269899016783, 0.05501567646043786, 0.055021240135226926, 0.05504638818988694, 0.05488723769555776]


# tune lightgbm using optuna 

In [9]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from lightgbm import LGBMRegressor

# We'll use the same train/validation split for a fair comparison
X_train, X_val, y_train, y_val = train_test_split(X_aligned, y, test_size=0.2, random_state=42)

# Define the objective function for LightGBM
def lgbm_objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'rmse',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'verbosity':-1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, preds)
    
    return rmse

# Create and run the study for LightGBM
# We can use more trials here as LGBM is generally faster
study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(lgbm_objective, n_trials=25)

# Print the best results
print("\n--- Optuna Tuning Results for LightGBM ---")
print(f"Best trial RMSE: {study_lgbm.best_value}")
print("Best hyperparameters found:")
print(study_lgbm.best_params)

[I 2025-10-23 12:13:40,319] A new study created in memory with name: no-name-c140baa4-bf2f-49e4-b9c8-2177bd033bec


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 425
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.350000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

[I 2025-10-23 12:14:04,301] Trial 0 finished with value: 0.05592284960154414 and parameters: {'n_estimators': 422, 'learning_rate': 0.19323914521580185, 'num_leaves': 252, 'max_depth': 4, 'min_child_samples': 48, 'subsample': 0.9731300929731581, 'colsample_bytree': 0.8810681760508363}. Best is trial 0 with value: 0.05592284960154414.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.113679 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 425
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.350000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

[I 2025-10-23 12:14:25,529] Trial 1 finished with value: 0.05601377058203294 and parameters: {'n_estimators': 294, 'learning_rate': 0.10582237223038163, 'num_leaves': 28, 'max_depth': 5, 'min_child_samples': 84, 'subsample': 0.8574422394921526, 'colsample_bytree': 0.9139696414388997}. Best is trial 0 with value: 0.05592284960154414.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027551 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 425
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.350000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

[I 2025-10-23 12:31:07,785] Trial 23 finished with value: 0.0558349753439299 and parameters: {'n_estimators': 822, 'learning_rate': 0.0350019627790779, 'num_leaves': 62, 'max_depth': 6, 'min_child_samples': 36, 'subsample': 0.8450370097906997, 'colsample_bytree': 0.9961721727677458}. Best is trial 13 with value: 0.05583413350081205.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021752 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 425
[LightGBM] [Info] Number of data points in the train set: 494203, number of used features: 18
[LightGBM] [Info] Start training from score 0.350000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

[I 2025-10-23 12:31:46,950] Trial 24 finished with value: 0.056868529739644784 and parameters: {'n_estimators': 815, 'learning_rate': 0.03924522106881859, 'num_leaves': 23, 'max_depth': 3, 'min_child_samples': 36, 'subsample': 0.9571600295752692, 'colsample_bytree': 0.9930271689776863}. Best is trial 13 with value: 0.05583413350081205.



--- Optuna Tuning Results for LightGBM ---
Best trial RMSE: 0.05583413350081205
Best hyperparameters found:
{'n_estimators': 697, 'learning_rate': 0.051879103733355705, 'num_leaves': 71, 'max_depth': 7, 'min_child_samples': 100, 'subsample': 0.9163669943797795, 'colsample_bytree': 0.7248666096727089}


# lightgbm tuned model

In [10]:
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# These are the best parameters we just found for LightGBM
tuned_lgbm_params = {
    'n_estimators': 697,
    'learning_rate': 0.051879103733355705,
    'num_leaves': 71,
    'max_depth': 7,
    'min_child_samples': 100,
    'subsample': 0.9163669943797795,
    'colsample_bytree': 0.7248666096727089,
    'objective': 'regression_l1',
    'metric': 'rmse',
    'verbosity': -1, # To keep the output clean
    'random_state': 42
}

# Set up the K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("--- Starting Cross-Validation for Tuned LightGBM Model ---")

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
    print(f"--- Fold {fold+1} ---")
    
    X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Initialize the model WITH OUR TUNED PARAMETERS
    tuned_model = LGBMRegressor(**tuned_lgbm_params)
    tuned_model.fit(X_train, y_train)
    
    y_pred = tuned_model.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    
    rmse_scores.append(rmse)
    print(f"RMSE for Fold {fold+1}: {rmse}")

# Calculate the final average CV score
print(f"\nAverage CV RMSE for Tuned LightGBM: {rmse_scores}")

--- Starting Cross-Validation for Tuned LightGBM Model ---
--- Fold 1 ---
RMSE for Fold 1: 0.0558437360248803
--- Fold 2 ---
RMSE for Fold 2: 0.05526645447244881
--- Fold 3 ---
RMSE for Fold 3: 0.05522965799627945
--- Fold 4 ---
RMSE for Fold 4: 0.05531171910012096
--- Fold 5 ---
RMSE for Fold 5: 0.05515856111494872

Average CV RMSE for Tuned LightGBM: [0.0558437360248803, 0.05526645447244881, 0.05522965799627945, 0.05531171910012096, 0.05515856111494872]


# blended model with 60/40 weight

In [13]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Our best parameters for both models
tuned_xgb_params = {
    'n_estimators': 617, 'learning_rate': 0.01636, 'max_depth': 8,
    'subsample': 0.854, 'colsample_bytree': 0.747, 'random_state': 42
}

tuned_lgbm_params = {
    'n_estimators': 697, 'learning_rate': 0.0518, 'num_leaves': 71,
    'max_depth': 7, 'min_child_samples': 100, 'subsample': 0.916,
    'colsample_bytree': 0.724, 'verbosity': -1, 'random_state': 42
}

# Set up K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
final_blend_scores = []

print("--- Starting Cross-Validation for FINAL TUNED BLEND ---")

for fold, (train_index, val_index) in enumerate(kf.split(X_aligned, y)):
    print(f"--- Fold {fold+1} ---")
    
    X_train, X_val = X_aligned.iloc[train_index], X_aligned.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # Train both TUNED models
    xgb_model = XGBRegressor(**tuned_xgb_params).fit(X_train, y_train)
    lgbm_model = LGBMRegressor(**tuned_lgbm_params).fit(X_train, y_train)
    
    # Blend predictions with a 60/40 weighted average
    xgb_preds = xgb_model.predict(X_val)
    lgbm_preds = lgbm_model.predict(X_val)
    blended_preds = (xgb_preds * 0.7) + (lgbm_preds * 0.3) # Weighted blend
    
    # Calculate RMSE
    rmse = root_mean_squared_error(y_val, blended_preds)
    final_blend_scores.append(rmse)
    print(f"Final Blended RMSE for Fold {fold+1}: {rmse}")

print(f"\nAverage CV RMSE for FINAL TUNED BLEND: {final_blend_scores}")

--- Starting Cross-Validation for FINAL TUNED BLEND ---
--- Fold 1 ---
Final Blended RMSE for Fold 1: 0.05556204020392719
--- Fold 2 ---
Final Blended RMSE for Fold 2: 0.05499564077627466
--- Fold 3 ---
Final Blended RMSE for Fold 3: 0.05499726404054736
--- Fold 4 ---
Final Blended RMSE for Fold 4: 0.055030209246574965
--- Fold 5 ---
Final Blended RMSE for Fold 5: 0.0548697820449616

Average CV RMSE for FINAL TUNED BLEND: [0.05556204020392719, 0.05499564077627466, 0.05499726404054736, 0.055030209246574965, 0.0548697820449616]


# 5. submission file

In [12]:
# --- Create ULTIMATE Submission File ---

# Our best parameters for both models
tuned_xgb_params = {
    'n_estimators': 617, 'learning_rate': 0.01636, 'max_depth': 8,
    'subsample': 0.854, 'colsample_bytree': 0.747, 'random_state': 42
}

tuned_lgbm_params = {
    'n_estimators': 697, 'learning_rate': 0.0518, 'num_leaves': 71,
    'max_depth': 7, 'min_child_samples': 100, 'subsample': 0.916,
    'colsample_bytree': 0.724, 'verbosity': -1, 'random_state': 42
}

# 1. Train the final TUNED models on ALL data
print("--- Training Final Tuned Models on All Data ---")
final_xgb = XGBRegressor(**tuned_xgb_params).fit(X_aligned, y)
final_lgbm = LGBMRegressor(**tuned_lgbm_params).fit(X_aligned, y)
print("Training complete!")

# 2. Make predictions on the test set
xgb_test_preds = final_xgb.predict(test_aligned)
lgbm_test_preds = final_lgbm.predict(test_aligned)

# 3. Blend the final predictions with our 60/40 weighted average
final_blended_preds = (xgb_test_preds * 0.6) + (lgbm_test_preds * 0.4)

# 4. Create the submission file
submission_df = pd.DataFrame({'id': test_ids, 'accident_risk': final_blended_preds})
submission_df.to_csv('final_tuned_submission.csv', index=False)

print("\nUltimate submission file 'final_tuned_submission.csv' created successfully!")
print(submission_df.head())

--- Training Final Tuned Models on All Data ---
Training complete!

Ultimate submission file 'final_tuned_submission.csv' created successfully!
       id  accident_risk
0  517754       0.292822
1  517755       0.121337
2  517756       0.183008
3  517757       0.318638
4  517758       0.389975
